In [2]:
import os
import pandas as pd


In [5]:
wb = pd.read_csv("data/cleaned/worldbank_clean.csv").convert_dtypes()
imf = pd.read_csv("data/cleaned/imf_clean.csv").convert_dtypes()
oecd = pd.read_csv("data/cleaned/oecd_clean.csv").convert_dtypes()
print(wb.shape)
print(imf.shape)
print(oecd.shape)

(1018, 4)
(1007, 3)
(48, 6)


In [6]:
wb = wb.rename(columns={"Country Name": "Country"})
wb.head()

,Country,Country Code,Year,TertiaryEnrollment
0,Aruba,ABW,2024,14.065542
1,Afghanistan,AFG,2018,9.99592
2,Afghanistan,AFG,2020,10.85436
3,Angola,AGO,2023,10.049358
4,Angola,AGO,2024,9.954907


In [11]:
merged = pd.merge(wb, imf, how="outer", on=["Country", "Year"], indicator=True)
merged["_merge"].value_counts()

_merge
both          746
left_only     272
right_only    261
Name: count, dtype: int64

In [12]:
merged = pd.merge(merged, oecd, how="left", on=["Country", "Year"])
merged[merged["Year"] == 2023].head()

,Country,Country Code,Year,TertiaryEnrollment,UnemploymentRate,_merge,qc_flag,attain_below_upper_sec,attain_upper_sec,attain_tertiary
8,Albania,ALB,2023,64.729352,9.5,both,<NA>,<NA>,<NA>,<NA>
17,Algeria,DZA,2023,55.532019,<NA>,left_only,<NA>,<NA>,<NA>,<NA>
25,Andorra,AND,2023,62.410694,1.525,both,<NA>,<NA>,<NA>,<NA>
28,Angola,AGO,2023,10.049358,<NA>,left_only,<NA>,<NA>,<NA>,<NA>
36,Argentina,ARG,2023,107.822574,6.125,both,ok,32.177174,43.569042,23.67523


In [13]:
matched = merged[merged["_merge"] == "both"]
unmatched = merged[merged["_merge"] != "both"]

In [20]:
oecd_unmatched = oecd[~oecd["Country"].isin(matched[matched["Year"] == 2023]["Country"])]

print("Matched records:", len(matched))
print("Unmatched records:", len(unmatched))
print("OECD countries not linked:", oecd_unmatched["Country"].tolist())

Matched records: 746
Unmatched records: 533
OECD countries not linked: ['Brazil', 'Colombia', 'Costa Rica', 'Ireland', 'Peru', 'United States']


In [21]:
os.makedirs("data/integrated", exist_ok=True)

merged.to_csv("data/integrated/integrated_full.csv", index=False)
matched.to_csv("data/integrated/integrated_matched.csv", index=False)
unmatched.to_csv("data/integrated/unmatched.csv", index=False)

In [ ]:
# Post-Integration Quality Assessment
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

display(Markdown('## Post-Integration Quality Assessment'))
keys = ['Country', 'Year']
clean = Path('data/cleaned')
out = Path('data/integrated')
wb_qa = pd.read_csv(clean / 'worldbank_clean.csv').rename(columns={'Country Name': 'Country'})
imf_qa = pd.read_csv(clean / 'imf_clean.csv')
oecd_qa = pd.read_csv(clean / 'oecd_clean.csv')
full_qa = pd.read_csv(out / 'integrated_full.csv')
matched_qa = pd.read_csv(out / 'integrated_matched.csv')
unmatched_qa = pd.read_csv(out / 'unmatched.csv')

# Consistency: verify that all country-year keys are present and unique.
for name, table in [('World Bank', wb_qa), ('IMF', imf_qa), ('OECD', oecd_qa), ('Full output', full_qa)]:
    assert table[keys].notna().all().all(), f'Missing country or year in {name}'
    assert not table.duplicated(keys).any(), f'Duplicate country-year pair in {name}'
print('PASS: All country-year keys are present and unique.')

# Accuracy: compare the saved files with the cleaned source values.
expected = wb_qa.merge(imf_qa, on=keys, how='outer', indicator=True, validate='one_to_one')
expected = expected.merge(oecd_qa, on=keys, how='left', validate='one_to_one')
expected['_merge'] = expected['_merge'].astype(str)

def compare_records(actual, expected_rows):
    pd.testing.assert_frame_equal(
        actual.sort_values(keys).reset_index(drop=True),
        expected_rows.sort_values(keys).reset_index(drop=True),
        check_dtype=False, check_like=True,
    )

compare_records(full_qa, expected)
compare_records(matched_qa, full_qa.loc[full_qa['_merge'] == 'both'])
compare_records(unmatched_qa, full_qa.loc[full_qa['_merge'] != 'both'])
print('PASS: The full, matched, and unmatched files agree with the cleaned inputs.')

# Completeness and timeliness: show which measures are available each year.
both = full_qa[['TertiaryEnrollment', 'UnemploymentRate']].notna().all(axis=1)
all_three = both & full_qa['attain_tertiary'].notna()
print(f'{len(full_qa):,} total country-year records; {both.sum()} have enrollment and unemployment; {all_three.sum()} also have tertiary attainment.')
display(Markdown('### Coverage by year'))
coverage = pd.DataFrame({'Year': full_qa['Year'], 'Enrollment and unemployment': both, 'All three measures': all_three})
display(coverage.groupby('Year').sum())

display(Markdown('### OECD records flagged for review'))
flagged = full_qa['qc_flag'].notna() & full_qa['qc_flag'].ne('ok')
display(full_qa.loc[flagged, ['Country', 'Year', 'qc_flag']])
display(Markdown('Annual coverage varies, so we will report the number of countries included in each comparison. The OECD year of 2023 is provisional. We need to confirm the reference year before interpreting attainment and unemployment as same-year observations.'))